# Thesis: Entity-Aware A-RAG with Evidence Verification

**Đề tài**: Nghiên cứu cải tiến mô hình A-RAG dựa trên theo dõi thực thể và kiểm chứng bằng chứng trong hỏi đáp đa bước

**Branch**: `thesis-entity-evidence-arag`

**Thứ tự chạy**: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11

## Cell 1: Clone Repo / Pull Code

In [1]:
import os, subprocess

REPO_URL = "https://github.com/trangdx2602/arag.git"
DATA_URL = "https://huggingface.co/datasets/Ayanami0730/rag_test"
REPO_DIR = "/content/arag"
BRANCH = "thesis-entity-evidence-arag"
FORCE_RECLONE = False  # Set True nếu muốn xóa và clone lại từ đầu

# --- Clone / pull repo ---
if FORCE_RECLONE and os.path.exists(REPO_DIR):
    import shutil; shutil.rmtree(REPO_DIR)
    print("Removed existing repo for re-clone.")

if os.path.exists(REPO_DIR):
    remote = subprocess.run(["git", "-C", REPO_DIR, "remote", "get-url", "origin"],
                            capture_output=True, text=True).stdout.strip()
    if "trangdx2602" not in remote:
        import shutil; shutil.rmtree(REPO_DIR)
        print(f"Wrong remote ({remote}), re-cloning from {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
        !cd {REPO_DIR} && git checkout {BRANCH}
    else:
        print("Repo already exists — pulling latest...")
        !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone {REPO_URL} {REPO_DIR}
    !cd {REPO_DIR} && git checkout {BRANCH}

# --- Download dataset from HuggingFace ---
DATA_DIR = f"{REPO_DIR}/data"
if not os.path.exists(f"{DATA_DIR}/musique/chunks.json"):
    print("Downloading dataset from HuggingFace (2-5 phút)...")
    !pip install huggingface_hub -q
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Ayanami0730/rag_test",
        repo_type="dataset",
        local_dir=DATA_DIR,
        ignore_patterns=["*.git*"],
    )
    print("Dataset downloaded.")
else:
    print("Dataset already present.")

!ls {REPO_DIR}

# Change working directory so relative paths in YAML configs resolve correctly
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/arag'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 122 (delta 31), reused 111 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 2.63 MiB | 16.30 MiB/s, done.
Resolving deltas: 100% (31/31), done.
Branch 'thesis-entity-evidence-arag' set up to track remote branch 'thesis-entity-evidence-arag' from 'origin'.
Switched to a new branch 'thesis-entity-evidence-arag'


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Dataset downloaded.
assets	      configs  docs	  pyproject.toml  scripts  tests
CITATION.cff  data     notebooks  README.md	  src
Working directory: /content/arag


## Cell 2: Cài Môi Trường + Set API Key

In [2]:
!pip install -e "/content/arag[full]" -q

import os

# ============================================================
# SET YOUR API KEY HERE (or use Colab Secrets)
# ============================================================
# Option A: Direct (not recommended for sharing)
# os.environ["ARAG_API_KEY"] = "sk-..."
# os.environ["ARAG_MODEL"] = "gpt-4o-mini"

# Option B: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ["ARAG_API_KEY"] = userdata.get("ARAG_API_KEY")
    os.environ["ARAG_MODEL"] = userdata.get("ARAG_MODEL") or "gpt-4o-mini"
    os.environ["ARAG_BASE_URL"] = userdata.get("ARAG_BASE_URL") or "https://api.openai.com/v1"
    print("API key loaded from Colab Secrets")
except Exception:
    print("WARNING: ARAG_API_KEY not set. Set it before running experiments.")

print("API key set:", bool(os.environ.get("ARAG_API_KEY")))
print("Model:", os.environ.get("ARAG_MODEL", "gpt-4o-mini"))
print("Base URL:", os.environ.get("ARAG_BASE_URL", "https://api.openai.com/v1"))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for arag (pyproject.toml) ... done
API key set: False
Model: gpt-4o-mini
Base URL: https://api.openai.com/v1


## Cell 3: Mount Google Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"
import os
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_RESULTS_DIR}")

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/thesis_arag_results


## Cell 4: Build Index HotpotQA / MuSiQue (GPU)

In [4]:
import os
REPO_DIR = "/content/arag"

# Build MuSiQue index
!python {REPO_DIR}/scripts/build_index.py \
    --chunks {REPO_DIR}/data/musique/chunks.json \
    --output {REPO_DIR}/data/musique/index \
    --model Qwen/Qwen3-Embedding-0.6B \
    --device cuda:0

# Build HotpotQA index
!python {REPO_DIR}/scripts/build_index.py \
    --chunks {REPO_DIR}/data/hotpotqa/chunks.json \
    --output {REPO_DIR}/data/hotpotqa/index \
    --model Qwen/Qwen3-Embedding-0.6B \
    --device cuda:0

print("Indexes built.")

Loading chunks from: /content/arag/data/musique/chunks.json
Loaded 1354 chunks
Extracting sentences...
Processing chunks: 100% 1354/1354 [00:00<00:00, 9371.71it/s]
Total sentences: 50767
Loading model: Qwen/Qwen3-Embedding-0.6B
modules.json: 100% 349/349 [00:00<00:00, 1.49MB/s]
config_sentence_transformers.json: 100% 215/215 [00:00<00:00, 936kB/s]
README.md: 100% 17.2k/17.2k [00:00<00:00, 30.3MB/s]
config.json: 100% 727/727 [00:00<00:00, 4.05MB/s]
model.safetensors: 100% 1.19G/1.19G [00:07<00:00, 160MB/s]
Loading weights: 100% 310/310 [00:00<00:00, 931.97it/s, Materializing param=norm.weight]
tokenizer_config.json: 100% 9.71k/9.71k [00:00<00:00, 26.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 61.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 110MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:00<00:00, 22.9MB/s]
config.json: 100% 313/313 [00:00<00:00, 2.10MB/s]
/content/arag/scripts/build_index.py:80: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `g

## Cell 5: Chạy Naive RAG

In [5]:
REPO_DIR = "/content/arag"
LIMIT = 20  # MVP: 20 câu. Đổi thành 100 khi expand
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_base.yaml \
    --variant naive_rag \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/naive_rag_musique \
    --limit {LIMIT} --workers {WORKERS}

print("Naive RAG done.")

Variant: naive_rag
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 234, in main
    shared_checker_llm = LLMClient(
                         ^^^^^^^^^^
  File "/content/arag/src/arag/core/llm.py", line 74, in __init__
    raise ValueError("API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.")
ValueError: API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.
Naive RAG done.


## Cell 6: Chạy A-RAG Baseline

In [6]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_base.yaml \
    --variant arag_baseline \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_baseline_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG Baseline done.")

Variant: arag_baseline
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 234, in main
    shared_checker_llm = LLMClient(
                         ^^^^^^^^^^
  File "/content/arag/src/arag/core/llm.py", line 74, in __init__
    raise ValueError("API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.")
ValueError: API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.
A-RAG Baseline done.


## Cell 7: Chạy A-RAG + Entity Tracker

In [7]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_entity.yaml \
    --variant arag_entity_tracker \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_entity_tracker_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Entity Tracker done.")

Variant: arag_entity_tracker
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 234, in main
    shared_checker_llm = LLMClient(
                         ^^^^^^^^^^
  File "/content/arag/src/arag/core/llm.py", line 74, in __init__
    raise ValueError("API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.")
ValueError: API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.
A-RAG + Entity Tracker done.


## Cell 8: Chạy A-RAG + Evidence Checker

In [8]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_evidence.yaml \
    --variant arag_evidence_checker \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_evidence_checker_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Evidence Checker done.")

Variant: arag_evidence_checker
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 234, in main
    shared_checker_llm = LLMClient(
                         ^^^^^^^^^^
  File "/content/arag/src/arag/core/llm.py", line 74, in __init__
    raise ValueError("API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.")
ValueError: API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.
A-RAG + Evidence Checker done.


## Cell 9: Chạy A-RAG + Full (ET + EV)

In [9]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_full.yaml \
    --variant arag_entity_evidence_full \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_entity_evidence_full_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG Full done.")

Variant: arag_entity_evidence_full
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 234, in main
    shared_checker_llm = LLMClient(
                         ^^^^^^^^^^
  File "/content/arag/src/arag/core/llm.py", line 74, in __init__
    raise ValueError("API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.")
ValueError: API key required. Set ARAG_API_KEY environment variable or pass api_key parameter.
A-RAG Full done.


## Cell 10: Evaluate và Xuất Bảng So Sánh

In [10]:
import subprocess, json
REPO_DIR = "/content/arag"

# Quick contain-match comparison (no LLM needed)
!python {REPO_DIR}/scripts/thesis/compare_results.py \
    --results {REPO_DIR}/results/thesis/ \
    --dataset musique

# Optional: LLM-based accuracy evaluation (costs money)
# for variant in ["naive_rag", "arag_baseline", "arag_entity_tracker",
#                 "arag_evidence_checker", "arag_entity_evidence_full"]:
#     !python {REPO_DIR}/scripts/eval.py \
#         --predictions {REPO_DIR}/results/thesis/{variant}_musique/predictions.jsonl \
#         --config {REPO_DIR}/configs/thesis/musique_base.yaml \
#         --workers 5

# Display comparison JSON
import json
cmp_file = f"{REPO_DIR}/results/thesis/comparison_musique.json"
try:
    with open(cmp_file) as f:
        cmp = json.load(f)
    import pandas as pd
    rows = []
    for v, stats in cmp.items():
        if stats:
            rows.append({"variant": v, **stats})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except Exception as e:
    print(f"Cannot display table: {e}")


=== Thesis Results Comparison — MUSIQUE ===

  naive_rag: no results found at /content/arag/results/thesis/musique_naive_rag/predictions.jsonl
  arag_baseline: no results found at /content/arag/results/thesis/musique_arag_baseline/predictions.jsonl
  arag_entity_tracker: no results found at /content/arag/results/thesis/musique_arag_entity_tracker/predictions.jsonl
  arag_evidence_checker: no results found at /content/arag/results/thesis/musique_arag_evidence_checker/predictions.jsonl
  arag_entity_evidence_full: no results found at /content/arag/results/thesis/musique_arag_entity_evidence_full/predictions.jsonl

---------------------------------------------------------------------------------------------------------------------------
Variant                       N       Contain-Acc   Avg Loops    Avg Tokens    Avg Cost($)   Avg Entities    Avg Coverage  
---------------------------------------------------------------------------------------------------------------------------
naive_r

## Cell 11: Copy Results về Google Drive

In [11]:
import shutil, os
REPO_DIR = "/content/arag"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"

src = f"{REPO_DIR}/results/thesis"
dst = f"{DRIVE_RESULTS_DIR}/results_thesis"

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Results copied to: {dst}")
else:
    print("No results to copy yet.")

# List saved files
for root, dirs, files in os.walk(dst):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path.replace(dst, '')} ({size:,} bytes)")

Results copied to: /content/drive/MyDrive/thesis_arag_results/results_thesis
  /comparison_musique.json (149 bytes)
